# Milestone 4 — Sequence Modeling with LSTM and GRU

This milestone introduces **deep learning models (LSTM / GRU)** that are specifically designed to capture the **order and contextual relationships** between words in a sequence.

---

##  Suggested Readings
- [LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html)
- [GRU](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html)

---

## ⚙️ Instructions

Use the **constants and helper functions** provided in the next cell to answer all **Milestone-4 questions**.

Perform the following tasks on the **training dataset** provided as part of the Kaggle competition:

🔗 **Competition Link:**  
[2025-Sep-DL-Gen-AI-Project](https://www.kaggle.com/competitions/2025-sep-dl-gen-ai-project)


# Imports

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
import random
from collections import Counter
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import warnings
warnings.filterwarnings("ignore")

import wandb
import time

### Set seeds and Constants

In [93]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
MAX_LEN = 50
BATCH_SIZE = 64
EMB_DIM = 100
HIDDEN_DIM = 256
OUTPUT_DIM = 5

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

# Create Vocab

In [3]:
data_path = '../data/train.csv'
df = pd.read_csv(data_path)             

In [4]:
# Split train df into train_df(80%) and test_df (20%) use seed
train_df, val_df = train_test_split(df, test_size=0.2, random_state=DATA_SEED)

In [5]:
# create a simple space-based tokenizer.
def tokenize(text):
    """Splits a text string into a list of tokens by space."""
    return text.split()

In [6]:
# Use counter to count all tokens in train_df
token_counter = Counter()

for text in train_df['text']:
    tokens = tokenize(text)
    token_counter.update(tokens)

print(f"Total unique tokens in training data: {len(token_counter)}")
print("Most common tokens:", token_counter.most_common(10))

Total unique tokens in training data: 10717
Most common tokens: [('my', 4497), ('i', 3798), ('the', 3293), ('and', 2828), ('to', 2351), ('a', 1841), ('was', 1431), ('of', 1269), ('in', 1244), ('it', 1014)]


## Create train and val dataloaders

In [8]:
#----------------------------- DON'T CHANGE THIS --------------------------
specials = ['<unk>', '<pad>']
min_freq = 2
vocab_list = specials + [token for token, freq in token_counter.items() if freq >= min_freq]
word2idx = {token: i for i, token in enumerate(vocab_list)}
def text_pipeline(text):
    """Converts text to a list of indices using the word2idx dict."""
    tokens = tokenize(text)
    return [word2idx.get(token, UNK_IDX) for token in tokens]
class EmotionDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe['text'].values
        self.labels = dataframe[['anger', 'fear', 'joy', 'sadness', 'surprise']].values.astype(np.float32)
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]
def collate_batch(batch):
    label_list, text_list = [], []
    for (_text, _labels) in batch:
        label_list.append(_labels)
        processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)[:MAX_LEN]
        text_list.append(processed_text)
    label_list = torch.tensor(label_list, dtype=torch.float32)
    text_list = pad_sequence(text_list, batch_first=True, padding_value=PAD_IDX)
    if text_list.shape[1] < MAX_LEN:
        pad_tensor = torch.full(
            (text_list.shape[0], MAX_LEN - text_list.shape[1]),
            PAD_IDX,
            dtype=torch.int64
        )
        text_list = torch.cat((text_list, pad_tensor), dim=1)

    return text_list, label_list

In [9]:
# Create train and val dataloaders
# Get the index for <unk> (unknown) and <pad> (padding) tokens
UNK_IDX = word2idx['<unk>']
PAD_IDX = word2idx['<pad>']

train_dataset = EmotionDataset(train_df)
val_dataset = EmotionDataset(val_df)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # No need to shuffle validation data
    collate_fn=collate_batch
)

print(f"Train DataLoader created with {len(train_dataloader)} batches.")
print(f"Validation DataLoader created with {len(val_dataloader)} batches.")

Train DataLoader created with 86 batches.
Validation DataLoader created with 22 batches.


### Q1. What are the vocabulary size, padding token index, and unknown token index for the above dataset?

In [10]:
VOCAB_SIZE = len(word2idx)
print(f"Vocabulary size (VOCAB_SIZE): {VOCAB_SIZE}")
print(f"Padding token index (PAD_IDX): {PAD_IDX}")
print(f"Unknown token index (UNK_IDX): {UNK_IDX}")

Vocabulary size (VOCAB_SIZE): 5730
Padding token index (PAD_IDX): 1
Unknown token index (UNK_IDX): 0


### Q2.What are the indices for the words "happy", "alone", and "sad" in the vocabulary?

In [11]:
idx_happy = word2idx.get('happy', UNK_IDX)
idx_alone = word2idx.get('alone', UNK_IDX)
idx_sad = word2idx.get('sad', UNK_IDX)

In [12]:
print(f"Index for 'happy': {idx_happy}")
print(f"Index for 'alone': {idx_alone}")
print(f"Index for 'sad': {idx_sad}")

Index for 'happy': 1578
Index for 'alone': 2525
Index for 'sad': 885


In [13]:
# Get one batch to test shapes
# take one batch as input here and store it in text_batch
text_batch, label_batch = next(iter(train_dataloader))
print(f"Original text batch shape: {text_batch.shape}")

emb_layer = nn.Embedding(VOCAB_SIZE, EMB_DIM)
embedded_batch = emb_layer(text_batch)

# Simple LSTM layer Output Shape (Use constants defined in 2nd cell)
# EMB_DIM = 100, HIDDEN_DIM = 256
lstm_layer = nn.LSTM(
    input_size=EMB_DIM,
    hidden_size=HIDDEN_DIM,
    batch_first=True  # This tells the LSTM the batch dim is first
)

# Pass the embedded batch through the LSTM
lstm_output, (lstm_hidden, lstm_cell) = lstm_layer(embedded_batch)

Original text batch shape: torch.Size([64, 50])


### Q3. What is the output shape of the Embedding layer?


In [14]:
print(f"Embedding layer output shape: {embedded_batch.shape}")

Embedding layer output shape: torch.Size([64, 50, 100])


### Q4. What will be output shape of simple LSTM layer

In [15]:
print(f"LSTM 'output' tensor shape: {lstm_output.shape}")

LSTM 'output' tensor shape: torch.Size([64, 50, 256])


### Q5. What is the 'hidden' state shape from a simple LSTM?

In [16]:
print(f"LSTM 'hidden' state shape: {lstm_hidden.shape}")

LSTM 'hidden' state shape: torch.Size([1, 64, 256])


### Q6. What is the 'hidden' state shape from a simple GRU?

In [17]:
# Define a simple GRU layer
gru_layer = nn.GRU(
    input_size=EMB_DIM,
    hidden_size=HIDDEN_DIM,
    batch_first=True
)

gru_output, gru_hidden = gru_layer(embedded_batch)

In [18]:
print(f"GRU 'hidden' state shape: {gru_hidden.shape}")

GRU 'hidden' state shape: torch.Size([1, 64, 256])


### Q7. What is the 'output' tensor shape from a bidirectional LSTM?

In [19]:
# Define a bidirectional LSTM layer
bi_lstm_layer = nn.LSTM(
    input_size=EMB_DIM,
    hidden_size=HIDDEN_DIM,
    batch_first=True,
    bidirectional=True  # The key change is here
)

bi_lstm_output, (bi_lstm_hidden, bi_lstm_cell) = bi_lstm_layer(embedded_batch)

In [20]:
print(f"Bidirectional LSTM 'output' tensor shape: {bi_lstm_output.shape}")

Bidirectional LSTM 'output' tensor shape: torch.Size([64, 50, 512])


### Q8. What is the 'hidden' state shape from a bidirectional LSTM?

In [21]:
print(f"Bidirectional LSTM 'hidden' state shape: {bi_lstm_hidden.shape}")

Bidirectional LSTM 'hidden' state shape: torch.Size([2, 64, 256])


### Q9. Create 3 sequential models using the (Simple & Bidirectional) LSTM and Stacked GRU (2 layers). For all models, follow this(Embedding layer → [LSTM / BiLSTM / Stacked GRU] → Linear layer) architecture. What will be the training parameters in all 3 cases? (LSTM, BiLSTM, Stacked GRU)

In [22]:
def count_parameters(model):
    """Counts the number of trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [25]:
# --- 1. Simple LSTM Model ---
class SimpleLSTMModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(
            emb_dim, 
            hidden_dim, 
            num_layers=1, 
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        # text = [batch size, seq len]
        embedded = self.embedding(text)
        # embedded = [batch size, seq len, emb dim]
        output, (hidden, cell) = self.lstm(embedded)
        # hidden = [1, batch size, hidden dim]
        prediction = self.fc(hidden.squeeze(0)) 
        return prediction

In [104]:
# --- 2. Bidirectional LSTM Model ---
class BidirectionalLSTMModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, output_dim, n_layers, dropout_prob):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(
            emb_dim, 
            hidden_dim, 
            num_layers=n_layers,
            dropout=dropout_prob,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        output, (hidden, cell) = self.lstm(embedded)
        # hidden = [num_layers*2, batch size, hidden dim] = [2, 64, 256]
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        # hidden = [batch size, hidden_dim * 2]
        prediction = self.fc(hidden)
        return prediction

In [27]:
# --- 3. Stacked GRU Model (2 layers) ---
class StackedGRUModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(
            emb_dim, 
            hidden_dim, 
            num_layers=2,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        output, hidden = self.gru(embedded)
        # hidden = [num_layers, batch size, hidden dim] = [2, 64, 256]
        prediction = self.fc(hidden[-1,:,:]) 
        return prediction

In [28]:
# Simple LSTM
simple_lstm = SimpleLSTMModel(VOCAB_SIZE, EMB_DIM, HIDDEN_DIM, OUTPUT_DIM)
print(f"Total parameters in Simple LSTM: {count_parameters(simple_lstm):,}")

# Bidirectional LSTM
bi_lstm = BidirectionalLSTMModel(VOCAB_SIZE, EMB_DIM, HIDDEN_DIM, OUTPUT_DIM)
print(f"Total parameters in Bidirectional LSTM: {count_parameters(bi_lstm):,}")

# Stacked GRU
stacked_gru = StackedGRUModel(VOCAB_SIZE, EMB_DIM, HIDDEN_DIM, OUTPUT_DIM)
print(f"Total parameters in Stacked GRU: {count_parameters(stacked_gru):,}")

Total parameters in Simple LSTM: 940,877
Total parameters in Bidirectional LSTM: 1,308,749
Total parameters in Stacked GRU: 1,243,981


### Q10. If you experimented with both LSTM and GRU models using the same hyperparameters, which one achieved a better peak Macro F1-score in your W&B logs?

In [30]:
wandb.login()

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

  ········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\91762\_netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [141]:
LEARNING_RATE = 1e-3
N_EPOCHS = 10
N_LAYERS = 1
DROPOUT = 0.4
MODEL_TYPE_NAME = "SimpleLSTM"
# MODEL_TYPE_NAME = "BidirectionalLSTM-drop-0.4"
# MODEL_TYPE_NAME = "StackedGRU"

In [95]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [96]:
def calculate_f1(preds, y_true, threshold=0.5):
    """Calculates Macro F1 score for multi-label classification."""
    y_pred_probs = torch.sigmoid(preds)
    y_pred = (y_pred_probs > threshold).int()
    
    y_pred_cpu = y_pred.cpu().numpy()
    y_true_cpu = y_true.cpu().numpy()
    
    return f1_score(y_true_cpu, y_pred_cpu, average='macro', zero_division=0)

In [97]:
def train_epoch(model, iterator, optimizer, criterion):
    """Handles one full training epoch."""
    model.train()
    epoch_loss = 0
    
    for (features, labels) in iterator:
        features = features.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        predictions = model(features)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    return epoch_loss / len(iterator)

In [98]:
def evaluate_epoch(model, iterator, criterion):
    """Handles one full evaluation epoch."""
    model.eval()
    epoch_loss = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for (features, labels) in iterator:
            features = features.to(device)
            labels = labels.to(device)
            
            predictions = model(features)
            loss = criterion(predictions, labels)
            
            epoch_loss += loss.item()
            all_preds.append(predictions)
            all_targets.append(labels)
    
    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    
    f1 = calculate_f1(all_preds, all_targets)
    return epoch_loss / len(iterator), f1

In [142]:
wandb.init(
    project="24f1002325-t32025",  
    entity="24f1002325-iit-madras",             
    name=f"{MODEL_TYPE_NAME}-lr{LEARNING_RATE}-epochs{N_EPOCHS}",
    config={
        "model_type": MODEL_TYPE_NAME,
        "learning_rate": LEARNING_RATE,
        "n_layers": N_LAYERS,
        "dropout": DROPOUT,
        "epochs": N_EPOCHS,
        "batch_size": BATCH_SIZE,
        "max_len": MAX_LEN,
        "embedding_dim": EMB_DIM,
        "hidden_dim": HIDDEN_DIM,
        "output_dim": OUTPUT_DIM,
        "vocab_size": VOCAB_SIZE
    }
)

In [143]:
model = SimpleLSTMModel(VOCAB_SIZE, EMB_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)
# model = BidirectionalLSTMModel(VOCAB_SIZE, EMB_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, DROPOUT).to(device)
# model = StackedGRUModel(VOCAB_SIZE, EMB_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)

print(f"Training model: {MODEL_TYPE_NAME}")
print(f"Total parameters: {count_parameters(model):,}")

Training model: SimpleLSTM
Total parameters: 1,833,037


In [144]:
wandb.config.update({"parameters": count_parameters(model)})

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [145]:
print("Training started...")
start_time = time.time()

for epoch in range(N_EPOCHS):
    epoch_start_time = time.time()
    
    train_loss = train_epoch(model, train_dataloader, optimizer, criterion)
    val_loss, val_f1 = evaluate_epoch(model, val_dataloader, criterion)
    
    epoch_time = time.time() - epoch_start_time
    
    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_time:.2f}s')
    print(f'\tTrain Loss: {train_loss:.3f}')
    print(f'\t Val. Loss: {val_loss:.3f} |  Val. Macro F1: {val_f1:.4f}')
    
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_macro_f1": val_f1,
        "epoch_time_secs": epoch_time
    })

total_time = time.time() - start_time
print(f"Training finished. Total time: {total_time:.2f}s")

Training started...
Epoch: 01 | Epoch Time: 1.31s
	Train Loss: 0.573
	 Val. Loss: 0.571 |  Val. Macro F1: 0.1429
Epoch: 02 | Epoch Time: 1.00s
	Train Loss: 0.568
	 Val. Loss: 0.569 |  Val. Macro F1: 0.1457
Epoch: 03 | Epoch Time: 0.99s
	Train Loss: 0.566
	 Val. Loss: 0.568 |  Val. Macro F1: 0.1503
Epoch: 04 | Epoch Time: 0.99s
	Train Loss: 0.565
	 Val. Loss: 0.567 |  Val. Macro F1: 0.1593
Epoch: 05 | Epoch Time: 0.98s
	Train Loss: 0.561
	 Val. Loss: 0.568 |  Val. Macro F1: 0.1653
Epoch: 06 | Epoch Time: 0.99s
	Train Loss: 0.551
	 Val. Loss: 0.548 |  Val. Macro F1: 0.2518
Epoch: 07 | Epoch Time: 0.99s
	Train Loss: 0.539
	 Val. Loss: 0.540 |  Val. Macro F1: 0.2805
Epoch: 08 | Epoch Time: 1.01s
	Train Loss: 0.522
	 Val. Loss: 0.534 |  Val. Macro F1: 0.2837
Epoch: 09 | Epoch Time: 1.02s
	Train Loss: 0.491
	 Val. Loss: 0.520 |  Val. Macro F1: 0.2761
Epoch: 10 | Epoch Time: 1.02s
	Train Loss: 0.446
	 Val. Loss: 0.508 |  Val. Macro F1: 0.3547
Training finished. Total time: 10.34s


In [146]:
wandb.summary["total_training_time_secs"] = total_time
wandb.finish()

epoch,▁▂▃▃▄▅▆▆▇█
epoch_time_secs,█▁▁▁▁▁▁▁▂▂
train_loss,████▇▇▆▅▃▁
val_loss,█████▆▅▄▂▁
val_macro_f1,▁▁▁▂▂▅▆▆▅█
epoch,10
epoch_time_secs,1.02184
total_training_time_secs,10.34239
train_loss,0.44578
val_loss,0.50775
val_macro_f1,0.35475


### Q11. Compare the total training time for your best sequential model against the simple averaging model from Milestone 3. How much longer (in minutes or percentage) did the more complex model (LSTM and GRU) take to train for the same number of epochs?

In [136]:
class SimpleAveragingModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, output_dim):
        super().__init__()
        
        self.embedding_bag = nn.EmbeddingBag(
            num_embeddings=vocab_size,
            embedding_dim=emb_dim,
            mode="mean",
            padding_idx=PAD_IDX
        )
        
        self.fc = nn.Linear(emb_dim, output_dim)

    def forward(self, text):       
        embedded_mean = self.embedding_bag(text)
        
        return self.fc(embedded_mean)
        
MODEL_TYPE_NAME = "SimpleAveragingModel"

model = SimpleAveragingModel(
    VOCAB_SIZE, 
    EMB_DIM, 
    OUTPUT_DIM
).to(device)

In [147]:
(10.34 - 3.06) / 10.34 * 100

70.40618955512572

### Q12. If you experimented with both LSTM and GRU models using the same hyperparameters, which one achieved a better peak Macro F1-score in your W&B logs?

### Q13 Based on your experiments, what was the most impactful hyperparameter you tuned for your sequential model (e.g., learning rate, hidden size, number of layers, dropout rate)?